# ONOS REST API Walkthrough

This notebook walks through the REST API step by step. You will reuse these patterns in `exercises/exercise.ipynb`:

1. read devices
2. read hosts
3. find the attachment switches
4. query the path
5. build a flow rule body
6. add a flow rule
7. check the rule you added
8. remove a flow rule

Each section is self-contained — you can run cells independently.

<details>
<summary><strong>New to Jupyter?</strong> Click to expand</summary>

- Run a cell: click it, then press **Shift+Enter**
- The `[*]` next to a cell means it is currently running
- Output appears below the cell when it finishes
- If a cell errors, fix it and re-run — earlier cells stay in memory

</details>

## 0. Setup

Keep Mininet and the ONOS CLI running in other terminals while you work through these cells.

Run the next cell once to import the libraries used throughout this notebook.

In [ ]:
import json
import requests

## 1. Read devices

**Goal**

See the same switch inventory that ONOS shows in `onos> devices`.

**What this cell does**

- calls `GET /devices`
- extracts the `devices` list from the JSON response
- prints the device ID, type, and availability for each switch


In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

response = requests.get(f'{base}/devices', auth=auth, timeout=5)
response.raise_for_status()
devices = response.json()['devices']

for device in devices:
    print(
        f"id={device['id']} type={device['type']} available={device['available']}"
    )

## 2. Read hosts

Before running the next cell, switch to the Mininet terminal and run:

```text
mininet> pingall
```

ONOS learns hosts only after it sees traffic from them.

**What this cell does**

- calls `GET /hosts`
- reads each host's `ipAddresses`
- reads each host's first attachment switch from `locations`


In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

response = requests.get(f'{base}/hosts', auth=auth, timeout=5)
response.raise_for_status()
hosts = response.json()['hosts']

for host in hosts:
    locations = host.get('locations', [])
    location = locations[0] if locations else None
    switch = location['elementId'] if location else '(no location)'
    print(f"id={host['id']} ips={host.get('ipAddresses', [])} switch={switch}")

## 3. Find the attachment switches

Before we can query a path, we need the switch IDs at the edge of the path.

This cell:

1. reads the host records for `10.0.0.1` and `10.0.0.2`
2. looks inside `locations`
3. prints the switch each host attaches to

If this cell says a host was not found, rerun `pingall` and try again.


In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'

response = requests.get(f'{base}/hosts', auth=auth, timeout=5)
response.raise_for_status()
hosts = response.json()['hosts']

src_host = None
dst_host = None
for host in hosts:
    ips = host.get('ipAddresses', [])
    if src_ip in ips:
        src_host = host
    if dst_ip in ips:
        dst_host = host

if src_host is None or dst_host is None:
    print('Host not found. Run pingall in Mininet first.')
else:
    src_device = src_host['locations'][0]['elementId']
    dst_device = dst_host['locations'][0]['elementId']
    print(f'src_switch={src_device}')
    print(f'dst_switch={dst_device}')

## 4. Query the path

Now we can ask ONOS for the path between those two switches.

This cell:

1. finds the same two attachment switches again
2. calls `GET /paths/<srcDevice>/<dstDevice>`
3. prints the first path that ONOS returns


In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'

response = requests.get(f'{base}/hosts', auth=auth, timeout=5)
response.raise_for_status()
hosts = response.json()['hosts']

src_host = None
dst_host = None
for host in hosts:
    ips = host.get('ipAddresses', [])
    if src_ip in ips:
        src_host = host
    if dst_ip in ips:
        dst_host = host

if src_host is None or dst_host is None:
    print('Host not found. Run pingall in Mininet first.')
else:
    src_device = src_host['locations'][0]['elementId']
    dst_device = dst_host['locations'][0]['elementId']

    response = requests.get(
        f'{base}/paths/{src_device}/{dst_device}',
        auth=auth,
        timeout=5,
    )
    response.raise_for_status()
    paths = response.json()['paths']

    if not paths:
        print('No path found.')
    else:
        first_path = paths[0]
        print(f"Path cost: {first_path.get('cost', 'unknown')}")
        for link in first_path['links']:
            print(
                f"{link['src']['device']}:{link['src']['port']} -> "
                f"{link['dst']['device']}:{link['dst']['port']}"
            )

## 5. Build the flow rule body

Look at one entry from the path output above:

```json
{
  "src": {"device": "of:0000000000000001", "port": "2"},
  "dst": {"device": "of:0000000000000002", "port": "1"}
}
```

This tells you:
- **which switch** to install the rule on: `link['src']['device']`
- **which port** to forward traffic out of: `link['src']['port']`

The JSON body below turns that into a rule ONOS can install.

**Important fields**

- `appId` tags the rule so we can find and remove it later
- `selector` says which packets match (here: IPv4 from `10.0.0.1` to `10.0.0.2`)
- `treatment` says what to do — `OUTPUT` on the port from the path link above

In [ ]:
app_id = 'org.onosproject.rest'

flow_rule = {
    'priority': 40000,
    'timeout': 0,
    'isPermanent': True,
    'appId': app_id,
    'treatment': {
        'instructions': [
            {'type': 'OUTPUT', 'port': '2'}  # '2' is link['src']['port'] from the path output above
        ]
    },
    'selector': {
        'criteria': [
            {'type': 'ETH_TYPE', 'ethType': '0x0800'},
            {'type': 'IPV4_SRC', 'ip': '10.0.0.1/32'},
            {'type': 'IPV4_DST', 'ip': '10.0.0.2/32'},
        ]
    },
}

flow_rule

## 6. Add the flow rule

To install the rule, we POST that JSON body to `/flows/<deviceId>`.

The next cell:

1. rebuilds the same flow rule body
2. POSTs it to `/flows/<deviceId>`
3. checks that ONOS accepted the request

> **Note** This installs one rule on one device. In the exercise, `install_path_rules` does the same POST for every link on the path — forward direction on the source switch, reverse direction on the destination switch, and edge rules for the host-facing ports at each end.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
device_id = 'of:0000000000000001'  # replace with the device ID from the path output above
app_id = 'org.onosproject.rest'

flow_rule = {
    'priority': 40000,
    'timeout': 0,
    'isPermanent': True,
    'appId': app_id,
    'treatment': {
        'instructions': [
            {'type': 'OUTPUT', 'port': '2'}  # '2' is link['src']['port'] from the path output above
        ]
    },
    'selector': {
        'criteria': [
            {'type': 'ETH_TYPE', 'ethType': '0x0800'},
            {'type': 'IPV4_SRC', 'ip': '10.0.0.1/32'},
            {'type': 'IPV4_DST', 'ip': '10.0.0.2/32'},
        ]
    },
}

response = requests.post(
    f'{base}/flows/{device_id}',
    auth=auth,
    json=flow_rule,
    timeout=5,
)
response.raise_for_status()
print(f"POST /flows/{device_id} returned HTTP {response.status_code}")

## 7. Check the rule you added

Now we confirm that the rule is really there.

This cell:

1. does `GET /flows/<deviceId>`
2. keeps only the flows whose `appId` matches our tag
3. prints those rules so you can see the full JSON ONOS stored


In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
device_id = 'of:0000000000000001'
app_id = 'org.onosproject.rest'

response = requests.get(f'{base}/flows/{device_id}', auth=auth, timeout=5)
response.raise_for_status()
app_flows = [
    flow for flow in response.json()['flows']
    if flow.get('appId') == app_id
]

if not app_flows:
    print(f"No flows found with appId={app_id}")
else:
    print(f"Found {len(app_flows)} flow(s) with appId={app_id}:")
    for flow in app_flows:
        print(json.dumps(flow, indent=2))
        print()

## 8. Remove the flows with this `appId`

Removing a rule is a two-step pattern:

1. list the flows on a device
2. keep only the ones whose `appId` matches our tag
3. delete those flow IDs one by one

> **In the exercise** `remove_rules_by_app_id` does exactly this loop across all switches. You will call it in `reroute_once` to clean up before installing a new path.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
device_id = 'of:0000000000000001'
app_id = 'org.onosproject.rest'

response = requests.get(f'{base}/flows/{device_id}', auth=auth, timeout=5)
response.raise_for_status()
app_flows = [
    flow for flow in response.json()['flows']
    if flow.get('appId') == app_id
]

if not app_flows:
    print(f"No flows found with appId={app_id}")
else:
    for flow in app_flows:
        print(f"deleting id={flow['id']} appId={flow.get('appId', '(no appId)')}")

    for flow in app_flows:
        response = requests.delete(
            f"{base}/flows/{device_id}/{flow['id']}",
            auth=auth,
            timeout=5,
        )
        response.raise_for_status()

    response = requests.get(f'{base}/flows/{device_id}', auth=auth, timeout=5)
    response.raise_for_status()
    remaining_app_flows = [
        flow for flow in response.json()['flows']
        if flow.get('appId') == app_id
    ]

    print(f"deleted {len(app_flows)} flow(s) with appId={app_id}")
    print(f"remaining flows with appId={app_id}: {len(remaining_app_flows)}")

## Next step

This notebook showed the REST patterns one section at a time.

Open `exercises/exercise.ipynb` in Jupyter to apply these patterns in the full rerouting exercise.